# Очистка таблиц Users, Items и Users_Items

В этом ноутбуке я привожу таблицы в нормальный вид, оставляю только нужные столбцы и строки.

## Импорт библиотек

In [308]:
import pandas as pd
import numpy as np
import os
import re
import scipy.stats as sts
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from pathlib import Path

## Загрузка данных

In [309]:
BASE_PATH = os.path.abspath("../../../Tables/BaseTable")

In [310]:
users_df = pd.read_csv(os.path.join(BASE_PATH, 'Users.csv'), sep=';', encoding='cp1251')
items_df = pd.read_csv(os.path.join(BASE_PATH, 'Items.csv'), sep=';', encoding='cp1251')
user_item_df = pd.read_csv(os.path.join(BASE_PATH, 'User_Items_test_month.csv'), sep=';', encoding='cp1251')
user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')

C:\Users\msgt0\AppData\Local\Temp\ipykernel_23036\3844528746.py:4: DtypeWarning: Columns (0: 	Телефон ) have mixed types. Specify dtype option on import or set low_memory=False.
  user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')


In [311]:
print("Изначальные размеры:")
print(f"Users: {users_df.shape}")
print(f"Items: {items_df.shape}")
print(f"User_Items: {user_item_df.shape}")
print(f"User_Item_year: {user_item_year_df.shape}")

Изначальные размеры:
Users: (2893, 18)
Items: (258, 8)
User_Items: (3133, 17)
User_Item_year: (39289, 19)


In [312]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 18 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Имя                                        2892 non-null   str    
 1   Телефон                                    2892 non-null   str    
 2   Email                                      997 non-null    str    
 3   Категории                                  58 non-null     str    
 4   Дата рождения                              2 non-null      str    
 5   Потратил, ?                                2892 non-null   str    
 6   Оплатил, ?                                 2892 non-null   str    
 7   Пол                                        6 non-null      str    
 8   Карта                                      0 non-null      float64
 9   Скидка                                     2892 non-null   float64
 10  Последний визит                    

In [313]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


In [314]:
user_item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3133 entries, 0 to 3132
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    1777 non-null   str    
 1   Специализация мастера           1777 non-null   str    
 2   Имя клиента                     1154 non-null   str    
 3   	Телефон                        1154 non-null   float64
 4   Email                           502 non-null    str    
 5   Время визита                    1777 non-null   str    
 6   Имя	 создателя                  1777 non-null   str    
 7   Дата создания                   1777 non-null   str    
 8   Статус                          1777 non-null   str    
 9   Источник                        1777 non-null   str    
 10  Категория услуги                3096 non-null   str    
 11  Название услуги                 3096 non-null   str    
 12  Стоимость, руб                  3096 non-null

In [315]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   	Телефон                        12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя	 создателя                  22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

Заранее изучив данные, выявлены необходимые операции:

__Для Users__

1) Вместо Имя+Телефон+Email ввести `id_user`, а Email удалить
2) Удалить столбцы Дата рождения, Потратил в руб; Карта; Скидка; Последний визит; Чаевые; Количество посещений; Комментарий; Дополнительный телефон; Согласен на получение рассылок; Согласен на обработку персональных данных т.к. они большинством своим не информативны либо полностью пустые
3) В столбце Категория есть несколько возможных значений я хочу:
- все пропуски и значения "Клиенты отщепенцев", "ПРЕДОПЛАТА" заменить на "обычный"
- удалить все экземпляры (строчки), где категория "Черный список"
- заменить значение "Постоянный" на "обычный" (т.к. в таблице только 1 такой экземпляр)
4) Удалить пропуски, если они останутся

__Для Items__

1) Удалить все услуги, где значение "Категория товара или услуги в продаже" = "Солярий" (слишком много посещений и направление никак не связано с остальными услугами)
2) Удалить все услуги, которые были оказаны слишком маленькое количество раз ("Количество оказанных услуг"<8 раз) или где количество уникальных клиентов слишком маленькое ("Уникальные клиенты" < 7).
3) По сути каждая строчка это и есть информация об уникальной услуге, но я бы добавил столбец `id_item` и сделал его PK

__Для User-Items__

1) Удалить все элементы связанные с солярием
2) Заменить имена столбцов на отношение сущьностей (то есть столбец "Имя", который относиться к клиенту назвать "Имя клиента" и т.п.)
3) Удалить Создал (Имя, Дата);Статус; Источник; Информация (Статус, Имя, Дата); Оплачено полностью; Комментарий; Категория (они не имеют смысла или нулевые)
4) Добавить `user_id` (для каждого клиента такой же как в таблице Users(то есть чтобы одному клиенту и там и там значился 1 и тот же id)), добавить `item_id` (по аналогии с `user_id`) и удалить "Email"
5) Объеденить некоторые услуги исходя из бизнесс логики 

__Для User-Items-Year__

1) Повторить pipline для User-Items, но на файле с годом  


## Обработка Users

#### Вводим user_id для каждого уникального пользователя, определяя его с помощью тройки Имя+телефон+Email

In [316]:
def clean_phone(phone):
    if pd.isna(phone):
        return np.nan
    phone_str = str(phone).replace('.0', '').replace('+7', '8')
    return phone_str.strip()

In [317]:
users_df["Телефон"] = users_df["Телефон"].apply(clean_phone)
users_df = users_df.drop_duplicates(subset=["Телефон"], keep="first")
users_df["id_user"] = pd.Categorical(users_df["Телефон"]).codes

#### Настройка столбца 'Категории'

In [318]:
users_df['Категории'] = users_df['Категории'].fillna('обычный')
users_df['Категории'] = users_df['Категории'].replace({
    'Клиенты отщепенцев': 'обычный',
    'ПРЕДОПЛАТА': 'обычный',
    'Постоянный': 'обычный'
})

In [319]:
users_df['Категории'].value_counts()

Категории
обычный          2881
Черный список      12
Name: count, dtype: int64

In [320]:
users_df = users_df[users_df['Категории'] != 'Черный список'].copy()
users_df['Категории'].value_counts()

Категории
обычный    2881
Name: count, dtype: int64

Оставляем столбец, т.к. в будущем возможно введение других категорий.

In [321]:
users_df = users_df[[
    'id_user', 'Имя', 'Телефон',
    'Категории', 'Оплатил, ?', 'Последний визит'
]].copy()
users_df = users_df.rename(columns={'Оплатил, ?': 'Оплачено в руб'})

In [322]:
users_df.info()

<class 'pandas.DataFrame'>
Index: 2881 entries, 0 to 2892
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2881 non-null   int16
 1   Имя              2880 non-null   str  
 2   Телефон          2880 non-null   str  
 3   Категории        2881 non-null   str  
 4   Оплачено в руб   2880 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 140.7 KB


#### Удалим пропуски и зафиксируем таблицу

In [323]:
users_clean = users_df.dropna(subset=['Последний визит']).copy()

In [324]:
users_clean.isna().sum()

id_user            0
Имя                0
Телефон            0
Категории          0
Оплачено в руб     0
Последний визит    0
dtype: int64

In [325]:
users_clean.info()

<class 'pandas.DataFrame'>
Index: 2711 entries, 0 to 2719
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2711 non-null   int16
 1   Имя              2711 non-null   str  
 2   Телефон          2711 non-null   str  
 3   Категории        2711 non-null   str  
 4   Оплачено в руб   2711 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 132.4 KB


#### Настроем типы данных

In [326]:
users_clean['Оплачено в руб'] = users_clean['Оплачено в руб'].astype(str).str.replace(',', '.').astype(float)

In [327]:
users_clean['Последний визит'] = pd.to_datetime(users_clean['Последний визит'], errors='coerce')

In [328]:
users_clean.dtypes

id_user                     int16
Имя                           str
Телефон                       str
Категории                     str
Оплачено в руб            float64
Последний визит    datetime64[us]
dtype: object

## Обработка Items

In [329]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


# Новая предобработка

#### Переименовываем столбцы 

In [330]:
items_df.columns = items_df.columns.str.strip()  # убираем пробелы в именах
items_df = items_df.rename(columns={
    "Выручка от продажи услуг, ?": "Выручка от продажи услуг, руб",
    "Ср. выручка с одного клиента, ?": "Ср. выручка с одного клиента, руб"
})

#### Настраиваем типизацию

In [331]:
def clean_money_col(series):
    return pd.to_numeric(series.str.replace(" ", "").str.replace(",", "."), errors="coerce")

def clean_percent_col(series):
    # убираем %, пробелы, заменяем запятые, приводим к числу
    return pd.to_numeric(
        series.str.replace(" ", "")
           .str.replace("%", "")
           .str.replace(",", "."),
        errors="coerce"
    )

items_df["Количество оказанных услуг"] = pd.to_numeric(
    items_df["Количество оказанных услуг"], errors="coerce"
).astype("Int64")

items_df["Доля от оказанных услуг в %"] = clean_percent_col(items_df["Доля от оказанных услуг в %"])
items_df["Выручка от продажи услуг, руб"] = clean_money_col(items_df["Выручка от продажи услуг, руб"])
items_df["% от общей выручки за услуги"] = clean_percent_col(items_df["% от общей выручки за услуги"])
items_df["Ср. выручка с одного клиента, руб"] = clean_money_col(items_df["Ср. выручка с одного клиента, руб"])
items_df["Уникальные клиенты"] = pd.to_numeric(
    items_df["Уникальные клиенты"], errors="coerce"
).astype("Int64")

In [332]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Название услуги или товара             258 non-null    str    
 1   Категория товара или услуги в продаже  258 non-null    str    
 2   Количество оказанных услуг             258 non-null    Int64  
 3   Доля от оказанных услуг в %            258 non-null    float64
 4   Выручка от продажи услуг, руб          258 non-null    float64
 5   % от общей выручки за услуги           258 non-null    float64
 6   Ср. выручка с одного клиента, руб      258 non-null    float64
 7   Уникальные клиенты                     258 non-null    Int64  
dtypes: Int64(2), float64(4), str(2)
memory usage: 16.8 KB


Удалчем услуги с солярием 

In [333]:
items_no_solar = items_df[
    items_df["Категория товара или услуги в продаже"] != "Солярий"
].copy()

In [334]:
items_filtered = items_no_solar[
    (items_no_solar["Количество оказанных услуг"] >= 6) &
    (items_no_solar["Уникальные клиенты"] >= 5)
].copy()

## Создаем items_dim

In [335]:
rename_map = {
    # брови
    "Коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "Коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "Укрепление акрилом": "Укрепление",
    "Укрепление гелем": "Укрепление",

    # покрытие
    "Укрепление цветным гелем": "Покрытие",
    "Покрытие гель лаком LUXIO, EMi, OneNail (руки)": "Покрытие",
    "Покрытие стойким лаком (ноги)": "Покрытие",
    "Покрытие база + топ (бесцветные)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (РУКИ)": "Покрытие",
    "Покрытие стойким лаком (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, EMi, OneNail (ноги)": "Покрытие",
    "Покрытие гель лаком EMi, OneNail (руки)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, E.Mi, OneNail (РУКИ) 1 НОГОТЬ": "Покрытие",
    "Покрытие гель лаком с эффектом \"Кошачий глаз\" (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, OneNail ФРЕНЧ (РУКИ)": "Покрытие",

    # снятие
    "Снятие лака (РУКИ)": "Снятие",
    "Снятие гель-лака (1 ноготь)": "Снятие",
    "Снятие гель-лака": "Снятие",
    "Снятие стойкого лака (руки)": "Снятие",
    "Снятие лака (НОГИ)": "Снятие",
    "Снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [336]:
ITEM_COL = "Название услуги или товара"

items_df[ITEM_COL] = items_df[ITEM_COL].replace(rename_map)

#### Добавим id_item

In [337]:
items_dim_cols = [
    "Название услуги или товара",
    "Категория товара или услуги в продаже"
]

# присвоим id_item (сквозной integer, как в твоей схеме)
items_dim = (items_filtered[items_dim_cols]
    .reset_index(drop=True)
    .reset_index()  # index = 0..N-1
    .rename(columns={"index": "id_item"})
)

#### Добавляем флаги комплексов 

In [338]:
items_dim["is_complex"] = 0          # 0 = базовая, 1 = комплекс
items_dim["complex_level"] = 0       # 0 = нет, 1 = мини, 2 = полный

In [339]:
items_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 5 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   id_item                                155 non-null    int64
 1   Название услуги или товара             155 non-null    str  
 2   Категория товара или услуги в продаже  155 non-null    str  
 3   is_complex                             155 non-null    int64
 4   complex_level                          155 non-null    int64
dtypes: int64(3), str(2)
memory usage: 6.2 KB


In [340]:
complex_names = [
    "Маникюр комплекс (полный)",
    "Маникюр комплекс (мини) 1",
    "Маникюр комплекс (мини) 2",
    "Педикюр комплекс (полный)",
    "Педикюр комплекс (мини) 1",
    "Педикюр комплекс (мини) 2",
    "Окрашивание и коррекция бровей (комплекс)",
    "Наращивание ресниц (комплекс)",
]

# категории этих комплексов (подставь свои, я возьму пример)
complex_cats = {
    "Маникюр комплекс (полный)": "Маникюр",
    "Маникюр комплекс (мини) 1": "Маникюр",
    "Маникюр комплекс (мини) 2": "Маникюр",
    "Педикюр комплекс (полный)": "Педикюр",
    "Педикюр комплекс (мини) 1": "Педикюр",
    "Педикюр комплекс (мини) 2": "Педикюр",
    "Окрашивание и коррекция бровей (комплекс)": "Брови",
    "Наращивание ресниц (комплекс)": "Ресницы",
}

In [341]:
# максимальный id_item, с которого продолжаем нумерацию
max_id = items_dim["id_item"].max()

new_complexes = []
for i, name in enumerate(complex_names):
    new_complexes.append({
        "id_item": max_id + i + 1,
        "Название услуги или товара": name,
        "Категория товара или услуги в продаже": complex_cats[name],
        "is_complex": 1,
        "complex_level": 2 if "комплекс (полный)" in name else 1,
    })

df_complexes = pd.DataFrame(new_complexes)

In [342]:
df_complexes.shape

(8, 5)

In [343]:
items_dim = pd.concat([items_dim, df_complexes], ignore_index=True)

In [344]:
items_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 5 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   id_item                                163 non-null    int64
 1   Название услуги или товара             163 non-null    str  
 2   Категория товара или услуги в продаже  163 non-null    str  
 3   is_complex                             163 non-null    int64
 4   complex_level                          163 non-null    int64
dtypes: int64(3), str(2)
memory usage: 6.5 KB


## Создание items_stats

In [345]:
items_stats_cols = [
    "id_item",
    "Название услуги или товара",
    "Категория товара или услуги в продаже",
    "Количество оказанных услуг",
    "Доля от оказанных услуг в %",
    "Выручка от продажи услуг, руб",
    "% от общей выручки за услуги",
    "Ср. выручка с одного клиента, руб",
    "Уникальные клиенты",
]

In [346]:
items_df["id_item"] = range(1, len(items_df) + 1)

items_stats = items_df[items_stats_cols].copy()

Упорядочиваем

In [347]:
items_stats = items_stats.sort_values("id_item").reset_index(drop=True)

In [348]:
items_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id_item                                258 non-null    int64  
 1   Название услуги или товара             258 non-null    str    
 2   Категория товара или услуги в продаже  258 non-null    str    
 3   Количество оказанных услуг             258 non-null    Int64  
 4   Доля от оказанных услуг в %            258 non-null    float64
 5   Выручка от продажи услуг, руб          258 non-null    float64
 6   % от общей выручки за услуги           258 non-null    float64
 7   Ср. выручка с одного клиента, руб      258 non-null    float64
 8   Уникальные клиенты                     258 non-null    Int64  
dtypes: Int64(2), float64(4), int64(1), str(2)
memory usage: 18.8 KB


## Обработка User-Items-Year

Удаляем пустые столбцы 

In [349]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   	Телефон                        12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя	 создателя                  22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

In [350]:
user_item_year_df = user_item_year_df.loc[:, ~user_item_year_df.columns.str.contains("^Unnamed")]

Почистить имена колонок 

In [351]:
user_item_year_df.columns = user_item_year_df.columns.str.strip()

In [352]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   Телефон                         12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя	 создателя                  22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

In [353]:
COL_MASTER = "Имя мастера"
COL_MASTER_SPEC = "Специализация мастера"
COL_CLIENT_NAME = "Имя клиента"
COL_PHONE = "Телефон"
COL_EMAIL = "Email"
COL_VISIT_TIME = "Время визита"
COL_CATEGORY = "Категория услуги"
COL_SERVICE = "Название услуги"
COL_PRICE_DISCOUNT = "Стоимость с учетом скидки, руб"

In [354]:
def clean_money(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", ".", regex=False),
        errors="coerce"
    )

Приведение типов данных 

In [355]:
user_item_year_df[COL_PRICE_DISCOUNT] = clean_money(user_item_year_df[COL_PRICE_DISCOUNT])

Удалим строки без услуг (пропуски)

In [356]:
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].notna()]
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].str.strip() != ""]

In [357]:
user_item_year_df.shape[0]

39075

Фильтрация по солярию и абонимаентам 

In [358]:
mask_not_solar = user_item_year_df[COL_CATEGORY] != "Солярий"
mask_not_sub = user_item_year_df[COL_CATEGORY] != "Услуги ПО АБОНЕМЕНТАМ"
user_item_year_df = user_item_year_df[mask_not_solar & mask_not_sub].copy()

In [359]:
user_item_year_df.shape[0]

28857

Ренейм (объединение item)

In [360]:
rename_map = {
    # брови
    "Коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "Коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "Укрепление акрилом": "Укрепление",
    "Укрепление гелем": "Укрепление",

    # покрытие (много вариантов гель-лака и лака)
    "Укрепление цветным гелем": "Покрытие",
    "Покрытие гель лаком LUXIO, EMi, OneNail (руки)": "Покрытие",
    "Покрытие стойким лаком (ноги)": "Покрытие",
    "Покрытие база + топ (бесцветные)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (РУКИ)": "Покрытие",
    "Покрытие стойким лаком (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, EMi, OneNail (ноги)": "Покрытие",
    "Покрытие гель лаком EMi, OneNail (руки)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, E.Mi, OneNail (РУКИ) 1 НОГОТЬ": "Покрытие",
    "Покрытие гель лаком с эффектом \"Кошачий глаз\" (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, OneNail ФРЕНЧ (РУКИ)": "Покрытие",

    # снятие
    "Снятие лака (РУКИ)": "Снятие",
    "Снятие гель-лака (1 ноготь)": "Снятие",
    "Снятие гель-лака": "Снятие",
    "Снятие стойкого лака (руки)": "Снятие",
    "Снятие лака (НОГИ)": "Снятие",
    "Снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [361]:
user_item_year_df[COL_SERVICE] = user_item_year_df[COL_SERVICE].replace(rename_map)

Заполнение пропусков (полупстых строк)

In [362]:
user_item_year_df = user_item_year_df.sort_values([COL_PHONE, COL_VISIT_TIME])

In [363]:
fill_cols = [col for col in user_item_year_df.columns if col != COL_SERVICE]
user_item_year_df[fill_cols] = user_item_year_df[fill_cols].ffill(axis=0)

In [364]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28857 entries, 37715 to 39286
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    28857 non-null  str    
 1   Специализация мастера           28857 non-null  str    
 2   Имя клиента                     28857 non-null  str    
 3   Телефон                         28857 non-null  object 
 4   Email                           28857 non-null  str    
 5   Время визита                    28857 non-null  str    
 6   Имя	 создателя                  28857 non-null  str    
 7   Дата создания                   28857 non-null  str    
 8   Статус                          28857 non-null  str    
 9   Источник                        28857 non-null  str    
 10  Категория услуги                28857 non-null  str    
 11  Название услуги                 28857 non-null  str    
 12  Стоимость, руб                  28857 non-nu

Добавляем id_user

In [365]:
users_df["Телефон"] = users_df["Телефон"].astype(str).str.strip()
user_item_year_df[COL_PHONE] = user_item_year_df[COL_PHONE].astype(str).str.strip()

In [366]:
user_item_year_df["Телефон"] = user_item_year_df["Телефон"].apply(clean_phone)

In [367]:
user_item_year_df = user_item_year_df.merge(
    users_df[["id_user", "Телефон"]],
    on="Телефон",
    how="left"
)

In [368]:
user_item_year_df.isna().sum()

Имя\t мастера                      0
Специализация мастера              0
Имя клиента                        0
Телефон                            0
Email                              0
Время визита                       0
Имя\t создателя                    0
Дата создания                      0
Статус                             0
Источник                           0
Категория услуги                   0
Название услуги                    0
Стоимость, руб                     0
Стоимость с учетом скидки, руб     0
Оплачено полностью                 0
Комментарий                        0
Категория                         31
id_user                           31
dtype: int64

Удаляем пропуски по id_user

In [369]:
user_item_year_df = user_item_year_df[user_item_year_df["id_user"].notna() & user_item_year_df["Категория"].notna()].copy()
user_item_year_df["id_user"] = user_item_year_df["id_user"].astype("Int64")

In [370]:
user_item_year_df.isna().sum()

Имя\t мастера                     0
Специализация мастера             0
Имя клиента                       0
Телефон                           0
Email                             0
Время визита                      0
Имя\t создателя                   0
Дата создания                     0
Статус                            0
Источник                          0
Категория услуги                  0
Название услуги                   0
Стоимость, руб                    0
Стоимость с учетом скидки, руб    0
Оплачено полностью                0
Комментарий                       0
Категория                         0
id_user                           0
dtype: int64

Рефакторинг по комплексам 

In [371]:
MANICURE_SERVICES = [
    "Маникюр Комбинированный",
    "Маникюр Классический (обрезной)",
    "Маникюр Мужской аппаратный",
    "Маникюр Аппаратный",
    "Маникюр Мужской Классический",
    "Маникюр Европейский (необрезной)"
]

PEDICURE_SERVICES = [
    "Смарт Педикюр Мужской",
    "Педикюр Аппаратный",
    "Смарт Педикюр Женский",
    "Педикюр Классический",
    "Педикюр Мужской Комбинированный",
    "Педикюр Комбинированный"
]

LASH_EXT_SERVICES = [
    'Наращивание ресниц "1D"',
    'Наращивание ресниц «2,5D»',
    'Наращивание ресниц "1,5D"',
    'Наращивание ресниц "3D"',
    'Наращивание ресниц "2D"'
]

In [372]:
def process_visit(df_visit: pd.DataFrame) -> pd.DataFrame:
    services = df_visit[COL_SERVICE].tolist()

    def has_any(service_list):
        return any(s in services for s in service_list)

    has_cover = "Покрытие" in services
    has_remove = "Снятие" in services
    has_align = "Выравнивание ногтевой пластины" in services
    has_manicure = has_any(MANICURE_SERVICES)
    has_pedicure = has_any(PEDICURE_SERVICES)
    has_brow_corr = "Коррекция бровей/форма бровей" in services
    has_brow_henna = "Окрашивание бровей хной" in services
    has_brow_paint = "Окрашивание бровей / ресниц (краской)" in services
    has_lash_ext = has_any(LASH_EXT_SERVICES)
    has_lash_remove = "Снятие поресничное" in services

    to_drop_idx = set()
    new_rows = []

    def get_idx(name, multiple=True):
        idxs = df_visit.index[df_visit[COL_SERVICE] == name].tolist()
        return idxs if multiple else (idxs[0] if idxs else None)

    def get_idx_any(names, multiple=True):
        idxs = []
        for n in names:
            idxs.extend(df_visit.index[df_visit[COL_SERVICE] == n].tolist())
        return idxs if multiple else (idxs[0] if idxs else None)

    # Брови, полный комплекс
    if has_brow_corr and (has_brow_henna or has_brow_paint):
        corr_idx = get_idx("Коррекция бровей/форма бровей", multiple=False)
        color_name = "Окрашивание бровей хной" if has_brow_henna else "Окрашивание бровей / ресниц (краской)"
        color_idx = get_idx(color_name, multiple=False)
        if corr_idx is not None and color_idx is not None:
            to_drop_idx.update([corr_idx, color_idx])
            row = df_visit.loc[corr_idx].copy()
            row[COL_SERVICE] = "Окрашивание и коррекция бровей (комплекс)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[corr_idx, color_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Маникюр: полный
    if has_cover and has_remove and has_align and has_manicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        remove_idx = get_idx("Снятие", multiple=False)
        align_idx = get_idx("Выравнивание ногтевой пластины", multiple=False)
        man_idx = get_idx_any(MANICURE_SERVICES, multiple=False)
        if all(idx is not None for idx in [cover_idx, remove_idx, align_idx, man_idx]):
            to_drop_idx.update([cover_idx, remove_idx, align_idx, man_idx])
            row = df_visit.loc[man_idx].copy()
            row[COL_SERVICE] = "Маникюр комплекс (полный)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, remove_idx, align_idx, man_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Маникюр: мини 1
    elif has_cover and has_manicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        man_idx = get_idx_any(MANICURE_SERVICES, multiple=False)
        if cover_idx is not None and man_idx is not None:
            to_drop_idx.update([cover_idx, man_idx])
            row = df_visit.loc[man_idx].copy()
            row[COL_SERVICE] = "Маникюр комплекс (мини) 1"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, man_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Маникюр: мини 2
    if "Маникюр комплекс (полный)" not in [r[COL_SERVICE] for r in new_rows]:
        if has_cover and has_remove and has_manicure:
            cover_idx = get_idx("Покрытие", multiple=False)
            remove_idx = get_idx("Снятие", multiple=False)
            man_idx = get_idx_any(MANICURE_SERVICES, multiple=False)
            if cover_idx is not None and remove_idx is not None and man_idx is not None:
                if not ({cover_idx, remove_idx, man_idx} & to_drop_idx):
                    to_drop_idx.update([cover_idx, remove_idx, man_idx])
                    row = df_visit.loc[man_idx].copy()
                    row[COL_SERVICE] = "Маникюр комплекс (мини) 2"
                    row["Стоимость с учетом скидки, руб"] = (
                        df_visit.loc[[cover_idx, remove_idx, man_idx], "Стоимость с учетом скидки, руб"]
                        .sum(skipna=True)
                    )
                    new_rows.append(row)

    # Педикюр: полный
    if has_cover and has_remove and has_align and has_pedicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        remove_idx = get_idx("Снятие", multiple=False)
        align_idx = get_idx("Выравнивание ногтевой пластины", multiple=False)
        ped_idx = get_idx_any(PEDICURE_SERVICES, multiple=False)
        if all(idx is not None for idx in [cover_idx, remove_idx, align_idx, ped_idx]):
            to_drop_idx.update([cover_idx, remove_idx, align_idx, ped_idx])
            row = df_visit.loc[ped_idx].copy()
            row[COL_SERVICE] = "Педикюр комплекс (полный)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, remove_idx, align_idx, ped_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Педикюр: мини 1
    elif has_cover and has_pedicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        ped_idx = get_idx_any(PEDICURE_SERVICES, multiple=False)
        if cover_idx is not None and ped_idx is not None:
            to_drop_idx.update([cover_idx, ped_idx])
            row = df_visit.loc[ped_idx].copy()
            row[COL_SERVICE] = "Педикюр комплекс (мини) 1"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, ped_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Педикюр: мини 2
    if "Педикюр комплекс (полный)" not in [r[COL_SERVICE] for r in new_rows]:
        if has_cover and has_remove and has_pedicure:
            cover_idx = get_idx("Покрытие", multiple=False)
            remove_idx = get_idx("Снятие", multiple=False)
            ped_idx = get_idx_any(PEDICURE_SERVICES, multiple=False)
            if cover_idx is not None and remove_idx is not None and ped_idx is not None:
                if not ({cover_idx, remove_idx, ped_idx} & to_drop_idx):
                    to_drop_idx.update([cover_idx, remove_idx, ped_idx])
                    row = df_visit.loc[ped_idx].copy()
                    row[COL_SERVICE] = "Педикюр комплекс (мини) 2"
                    row["Стоимость с учетом скидки, руб"] = (
                        df_visit.loc[[cover_idx, remove_idx, ped_idx], "Стоимость с учетом скидки, руб"]
                        .sum(skipna=True)
                    )
                    new_rows.append(row)

    # Ресницы: полный комплекс
    if has_lash_ext and has_lash_remove:
        lash_idx = get_idx_any(LASH_EXT_SERVICES, multiple=False)
        rem_idx = get_idx("Снятие поресничное", multiple=False)
        if lash_idx is not None and rem_idx is not None:
            to_drop_idx.update([lash_idx, rem_idx])
            row = df_visit.loc[lash_idx].copy()
            row[COL_SERVICE] = "Наращивание ресниц (комплекс)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[lash_idx, rem_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    df_keep = df_visit.drop(index=list(to_drop_idx))
    if new_rows:
        df_new = pd.DataFrame(new_rows)
        return pd.concat([df_keep, df_new], ignore_index=True)
    return df_keep

In [373]:
user_item_year_df[COL_VISIT_TIME] = pd.to_datetime(
    user_item_year_df[COL_VISIT_TIME],
    errors="coerce"
)

In [374]:
group_cols = ["id_user", COL_VISIT_TIME]

processed_visits = []
for (uid, visit_time), df_visit in user_item_year_df.groupby(group_cols):
    processed = process_visit(df_visit)
    processed_visits.append(processed)

user_items_year_clean = pd.concat(processed_visits, ignore_index=True)

In [375]:
user_items_year_clean.shape

(28768, 18)

Добавление id_item

In [376]:
user_items_year_clean["Название услуги_norm"] = (
    user_items_year_clean["Название услуги"]
    .str.replace(" ",  "", regex=False)
    .str.replace(" ", "", regex=False)  # неразрывный пробел
    .str.strip()
    .str.lower()
)

items_dim["Название услуги или товара_norm"] = (
    items_dim["Название услуги или товара"]
    .str.replace(" ",  "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
    .str.lower()
)

In [377]:
user_items_year_clean = user_items_year_clean.merge(
    items_dim[["Название услуги или товара_norm", "id_item"]],
    left_on="Название услуги_norm",
    right_on="Название услуги или товара_norm",
    how="left"
)

# можно убрать временные колонки
user_items_year_clean = user_items_year_clean.drop(
    columns=["Название услуги_norm", "Название услуги или товара_norm"],
    errors="ignore"
)

In [378]:
user_items_year_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28768 entries, 0 to 28767
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Имя	 мастера                    28768 non-null  str           
 1   Специализация мастера           28768 non-null  str           
 2   Имя клиента                     28768 non-null  str           
 3   Телефон                         28768 non-null  str           
 4   Email                           28768 non-null  str           
 5   Время визита                    28768 non-null  datetime64[us]
 6   Имя	 создателя                  28768 non-null  str           
 7   Дата создания                   28768 non-null  str           
 8   Статус                          28768 non-null  str           
 9   Источник                        28768 non-null  str           
 10  Категория услуги                28768 non-null  str           
 11  Название услу

In [379]:
print("Пропусков id_item после нормализации:", user_items_year_clean["id_item"].isna().sum())

Пропусков id_item после нормализации: 11044


In [380]:
user_items_year_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28768 entries, 0 to 28767
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Имя	 мастера                    28768 non-null  str           
 1   Специализация мастера           28768 non-null  str           
 2   Имя клиента                     28768 non-null  str           
 3   Телефон                         28768 non-null  str           
 4   Email                           28768 non-null  str           
 5   Время визита                    28768 non-null  datetime64[us]
 6   Имя	 создателя                  28768 non-null  str           
 7   Дата создания                   28768 non-null  str           
 8   Статус                          28768 non-null  str           
 9   Источник                        28768 non-null  str           
 10  Категория услуги                28768 non-null  str           
 11  Название услу

In [381]:
print("Пропусков id_item:", user_items_year_clean["id_item"].isna().sum())

Пропусков id_item: 11044


In [382]:
# 1) номера строк с пропуском
mask_nan = user_items_year_clean["id_item"].isna()

# 2) уникальные названия, которые не нашлись в items_dim
missing_services = user_items_year_clean.loc[mask_nan, "Название услуги"].unique()
print("Уникальные названия, для которых нет id_item (первые 50):")
print(missing_services[:50])

Уникальные названия, для которых нет id_item (первые 50):
<StringArray>
[                                                               'Снятие',
                                                              'Покрытие',
           'Энзимный пилинг лица (поверхностный) - Эксфолиант CHRISTINA',
                                         'Коррекция бровей/форма бровей',
                               'Лазерная Эпиляция (FOR WOMEN) - Ягодицы',
                                             'Сахарная эпиляция ягодицы',
                                             'Пирсинг / Прокол Уха Снаг',
                                                 'Пирсинг / Прокол Носа',
                           'Коллагеновое восстановление волос (длинные)',
                                              'Сахарная эпиляция бикини',
 'Smart лифт массаж (лицо, шейно-воротниковая зона, декольте) (80 мин.)',
                                 'Лазерная Эпиляция (FOR WOMEN) - Скулы',
                                        

In [385]:
print("Снятие:")
print(items_dim["Название услуги или товара"].str.contains("Снятие", case=False).sum())

print("Покрытие:")
print(items_dim["Название услуги или товара"].str.contains("Покрытие", case=False).sum())

print("Коррекция бровей:")
print(items_dim["Название услуги или товара"].str.contains("Коррекция бровей", case=False).sum())

print("Комплексы:")
print(
    items_dim["Название услуги или товара"]
    [items_dim["Название услуги или товара"].str.contains("комплекс", case=False)]
    .unique()
)

Снятие:
8
Покрытие:
13
Коррекция бровей:
3
Комплексы:
<StringArray>
['КОМПЛЕКС №3 (Ноги полностью (Голени, Колени, Бедра), Руки полностью (Плечо, Предплечье, Локоть), Подмышечные впадины, Бикини на выбор (глубокое или классическое))',
                                             'КОМПЛЕКС №2 (Ноги полностью (Голени, Колени, Бедра), Подмышечные впадины, Бикини на выбор (глубокое или классическое))',
                                                                              'КОМПЛЕКС №1 (Голени, Подмышечные впадины, Бикини на выбор (глубокое или классическое)',
                                                                                                                                                 'Чистка комплексная',
                                                                                                                                          'Маникюр комплекс (полный)',
                                                                                                 

======================================================

## Сортировка таблиц по id

In [383]:
# 1. users_clean по user_id
users_clean = users_clean.sort_values('id_user').reset_index(drop=True)

# 2. items_clean по item_id
items_clean = items_clean.sort_values('id_item').reset_index(drop=True)

# 3. user_items_final по user_id
user_item_final = user_items_final.sort_values('user_id').reset_index(drop=True)

NameError: name 'items_clean' is not defined

## Выгрузка таблиц

In [ ]:

''' 
users_clean.to_csv('users_clean_final.csv', index=False, encoding='utf-8-sig')
items_clean.to_csv('items_clean_final.csv', index=False, encoding='utf-8-sig')
user_item_final.to_csv('user_items_final.csv', index=False, encoding='utf-8-sig')
'''